In [ ]:
import os
import multiprocessing as mp

def process_year(args):
    year, input_path, output_path, prefix = args
    os.system(f'cdo -setattribute,ch4@units="mgCH4/m2/d" -expr,"ch4=f_ch4_surf_flux_tot*16.04*1e3*86400" -selname,f_ch4_surf_flux_tot {input_path}/{prefix}_hist_{year}.nc {output_path}/net_methane_{year}_tmp1.nc')

def g2(input_path, output_path, area_path, prefix, yr_stt, yr_end):
    years = list(range(yr_stt, yr_end + 1))
    
    # Parallel processing of individual years
    with mp.Pool(processes=mp.cpu_count()) as pool:
        pool.map(process_year, [(year, input_path, output_path, prefix) for year in years])
    
    # Build the file list string for merging
    name = ' '.join([f"{output_path}net_methane_{year}_tmp1.nc" for year in years])
    
    os.system(f'cdo -O mergetime {name} {output_path}/net_methane_{yr_stt}_{yr_end}_tmp1.nc')
    os.system(f'cdo -O -monmean {output_path}/net_methane_{yr_stt}_{yr_end}_tmp1.nc {output_path}/net_methane_{yr_stt}_{yr_end}_tmp2.nc')
    os.system(f'cdo -O -yearmean {output_path}/net_methane_{yr_stt}_{yr_end}_tmp1.nc {output_path}/net_methane_{yr_stt}_{yr_end}_tmp3.nc')

    os.system(f'cdo -O -setattribute,ch4@units="gCH4/yr" -mulc,365 -divc,1e3 -mul {output_path}/net_methane_{yr_stt}_{yr_end}_tmp3.nc {area_path}Area_2p.nc {output_path}net_methane_{yr_stt}_{yr_end}_sum_tmp1.nc')
    os.system(f'cdo -O -setattribute,ch4@units="TgCH4/yr" -divc,1e12 -fldsum {output_path}net_methane_{yr_stt}_{yr_end}_sum_tmp1.nc {output_path}net_methane_{yr_stt}_{yr_end}_sum.nc')

prefix = 'g2'
input_path = f'/share/home/dq076/data/cases/globe/{prefix}/history/'
output_path = f'/share/home/dq076/data/cases/globe/{prefix}/postdata/'
area_path = f'/share/home/dq076/data/Area/'
os.makedirs(output_path, exist_ok=True)
g2(input_path, output_path, area_path, prefix, 1995, 2010)